What is Docker?

If you haven’t played with Docker yet, here’s the skinny: Docker lets you package up your apps and their entire environments into isolated containers. Think of containers like lightweight virtual machines, but way faster and more efficient.

Why is this important? Because it means you can run your app anywhere — your laptop, a cloud server, or even a colleague’s machine — and it’ll behave exactly the same. No more “works on my machine” headaches.

What About Docker Compose?

Docker Compose is like a remote control for your containers. Instead of running long, complicated docker run commands, you write a simple YAML file describing your containers, networks, and volumes. Then, with one command, you bring it all up or tear it down.

Use it for multi-container setups so I can control multiple containers with one file and spin them up and down and sideways and all around using one command: docker-compose up

Why Automate?

Manually typing commands every time you want to build images, create volumes, and spin up containers?

Automation ensures your environment is repeatable, consistent, and easily shareable. Plus, it saves you tons of time. Thus, 🧐 I used Python with the Docker SDK to automate building my Jenkins image, creating volumes, and running containers with the right settings — all in one script.

1. Writing the Docker Compose File
   
First, I wrote a docker-compose.yml to define the Jenkins container, mapped ports, and attached a Docker volume for data persistence. This got Jenkins running quickly and helped me experiment with persistent storage.

In [ ]:
services:
  jenkins:
    # pull jenkins LTS image from dockerhub
    image: jenkins/jenkins:lts
    ports:
      - "8080:8080"
    volumes:
        #create volume for jenkins to persist its data
      - jenkins_data:/var/jenkins_home
    restart: unless-stopped

volumes:
  jenkins_data:

2. Crafting the Custom Jenkins Dockerfile

Next, I created a Dockerfile to “customize” the Jenkins image. This way, every build of the image had exactly what I needed baked in (which was literally barebones Jenkins).

In [ ]:
FROM jenkins/jenkins:lts

USER root

RUN apt-get update && apt-get install -y \
    sudo \
    curl \
    git \
    && rm -rf /var/lib/apt/lists/*

USER jenkins

3. Automating with Python

Finally, I wrote a Python script (automate.py) using the Docker SDK to:

Build my custom Jenkins image from the Dockerfile.
Create and manage the persistent volume.
Run the Jenkins container with correct port mappings and volume mounts.
Retrieve and display the initial Jenkins admin password for easy access.
Running this script makes the entire setup a breeze, especially for repeated deployments or sharing with teammates.

In [ ]:
import docker  # type: ignore
import time

client = docker.from_env()

IMAGE_NAME = "cstu-jenkins"
VOLUME_NAME = "jenkins_data_cstu"
CONTAINER_NAME = "cstu-jenkins"
DOCKERFILE_PATH = "." 

# Step 2: Build the custom Jenkins image
print(f"Building image '{IMAGE_NAME}'...")
image, build_logs = client.images.build(path=DOCKERFILE_PATH, tag=IMAGE_NAME)
print("Image built successfully.")

# Step 3: Create volume
try:
    volume = client.volumes.get(VOLUME_NAME)
    print(f"Volume '{VOLUME_NAME}' already exists.")
except docker.errors.NotFound:
    volume = client.volumes.create(name=VOLUME_NAME)
    print(f"Volume '{VOLUME_NAME}' created.")

# Step 4: Run the container
print(f"Running container '{CONTAINER_NAME}'...")
try:
    container = client.containers.run(
        IMAGE_NAME,
        name=CONTAINER_NAME,
        ports={"8080/tcp": 8080},
        volumes={VOLUME_NAME: {'bind': '/var/jenkins_home', 'mode': 'rw'}},
        detach=True
    )
except docker.errors.APIError as e:
    print(f"Error: {e}")
    print("Trying to remove existing container and retry...")
    try:
        old_container = client.containers.get(CONTAINER_NAME)
        old_container.stop()
        old_container.remove()
        container = client.containers.run(
            IMAGE_NAME,
            name=CONTAINER_NAME,
            ports={"8080/tcp": 8080},
            volumes={VOLUME_NAME: {'bind': '/var/jenkins_home', 'mode': 'rw'}},
            detach=True
        )
    except Exception as e2:
        print(f"Failed to recover: {e2}")
        exit(1)

print("Container is starting. Waiting 15 seconds for Jenkins to initialize...")
time.sleep(15)  # Wait for Jenkins to generate password

# Step 5: Get the initial admin password
exec_log = container.exec_run("cat /var/jenkins_home/secrets/initialAdminPassword")
admin_password = exec_log.output.decode().strip()
print(f"\n  Jenkins Initial Admin Password:\n{admin_password}\n")